# BITS Pilani - M.Tech AIML - Machine Learning Assignment 2
## Dataset: Student Depression Dataset

This notebook performs preprocessing, model training for 5 classification models, evaluation across 6 key metrics, and exports saved model artifacts alongside preprocessed test data.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef

# Create directory for saved models
os.makedirs('model', exist_ok=True)

# 1. Load Dataset
df = pd.read_csv('Student_Depression_Dataset.csv')
print('Dataset Shape:', df.shape)
df.head()

Dataset Shape: (4464, 13)


,Gender,Age,Family_background,Relationship_status,Marital_status,Living_style,Current_semester,CGPA,friend_circle,Health_condition,Cost_without_study,Home_division,Depression
0,Female,22,Middle class,Single,Unmarried,Bachelor,8,3.43,Yes,Critical,10113,Chittagong,Yes
1,Male,19,Middle class,Single,Unmarried,Bachelor,2,3.70,Yes,Abnormal,13815,Chittagong,Yes
2,Male,23,Middle class,Single,Unmarried,Family,8,3.28,No,Abnormal,15450,Dhaka,Yes
3,Male,20,Middle class,Complicated,Unmarried,Family,2,3.77,No,Good,17413,Dhaka,No
4,Female,21,Middle class,Single,Unmarried,Family,1,4.00,No,Abnormal,16693,Dhaka,Yes


In [2]:
# 2. Data Preprocessing & Categorical Encoding
df_ml = df.copy()
df_ml['Depression'] = df_ml['Depression'].map({'Yes': 1, 'No': 0})

cat_cols = df_ml.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col])
    label_encoders[col] = le

X = df_ml.drop('Depression', axis=1)
y = df_ml['Depression']

# 3. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Export unscaled test dataset (csv format for Streamlit upload)
test_df = df.iloc[X_test.index].copy()
test_df.to_csv('test_data.csv', index=False)
print('Exported test_data.csv with', len(test_df), 'instances.')

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save preprocessing artifacts
joblib.dump(scaler, 'model/scaler.pkl')
joblib.dump(label_encoders, 'model/label_encoders.pkl')

C:\Users\pisce\AppData\Local\Temp\ipykernel_7672\1779088955.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_ml.select_dtypes(include=['object']).columns.tolist()


Exported test_data.csv with 893 instances.


['model/label_encoders.pkl']

In [3]:
# 4. Train 5 ML Models & Compute 6 Evaluation Metrics
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    results.append({
        'ML Model Name': name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })
    
    # Save model artifact
    model_path = f"model/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(model, model_path)
    print(f'Saved {name} to {model_path}')

metrics_df = pd.DataFrame(results)
print('\n--- Metrics Summary ---')
print(metrics_df.to_string(index=False))

Saved Logistic Regression to model/logistic_regression.pkl
Saved Decision Tree to model/decision_tree.pkl
Saved KNN to model/knn.pkl
Saved Naive Bayes to model/naive_bayes.pkl


C:\Users\pisce\AppData\Roaming\Python\Python312\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Saved Support Vector Machine to model/support_vector_machine.pkl

--- Metrics Summary ---
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.8555 0.9023     0.7681  0.6625 0.7114 0.6188
         Decision Tree    0.9922 0.9953     0.9874  0.9833 0.9854 0.9800
                   KNN    0.8903 0.9483     0.8480  0.7208 0.7793 0.7110
           Naive Bayes    0.8835 0.8961     0.8119  0.7375 0.7729 0.6963
Support Vector Machine    0.9026 0.9604     0.8844  0.7333 0.8018 0.7437
